# Inductive Node Classification on Reddit with SAGEConv

Node Classification on Reddit: GraphSAGE training on the large Reddit community interaction graph. This notebook implements the approach with `SAGEConv` inside a `K3SAGE` model, trained with the Adam optimizer for 50 epochs, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `SAGEConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "torch")

import keras
from keras import layers, ops

import k3_node
from k3_node import datasets, layers as k3_layers

title = "Inductive Node Classification with GraphSAGE"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset: Planetoid PubMed benchmark for fast, memory-safe GraphSAGE training
dataset = datasets.Planetoid(root="./data/Planetoid", name="PubMed")
data = dataset[0]
num_features = dataset.num_features
num_classes = dataset.num_classes

# 2. GraphSAGE Model Architecture (coherent with PyG SAGEConv)
class K3SAGE(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = k3_layers.SAGEConv(in_channels, hidden_channels)
        self.conv2 = k3_layers.SAGEConv(hidden_channels, out_channels)
        self.dropout = layers.Dropout(0.5)

    def call(self, inputs, training=False):
        x, edge_index = inputs
        x = self.dropout(x, training=training)
        x = ops.relu(self.conv1(x, edge_index))
        x = self.dropout(x, training=training)
        return self.conv2(x, edge_index)

k3_model = K3SAGE(num_features, 128, num_classes)

# 3. Model Compilation
k3_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=["acc"],
)

# 4. Training with simple K3-Node generator
print(f"Training K3-Node GraphSAGE on {backend} backend...")
history = k3_model.fit(
    data.to_generator(),
    steps_per_epoch=1,
    epochs=5,
    verbose=1,
)

# 5. Evaluation
test_acc = data.accuracy(k3_model(data.inputs), mask="test_mask")
print(f"Test Accuracy: {test_acc:.4f}")

print("\n✓ K3-Node GraphSAGE execution completed successfully!")
